In [66]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
import plotly.express as px
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report



pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
%config Completer.use_jedi = False

# Study Month

In [67]:
respeval = pd.read_csv("respeval.csv")

In [68]:
respeval['Study_month'] = pd.cut(respeval['DOSREFDY'], [respeval['DOSREFDY'].min(),0,31,61,91,121,151,181,210,respeval['DOSREFDY'].max()],labels = ['before_study','month1','month2','month3','month4','month5','month6','month7','>month7'])

In [69]:
respeval = respeval[['SUBJID','Study_month']]

In [70]:
respeval

,SUBJID,Study_month
0,201001001,before_study
1,201001001,before_study
2,201001001,before_study
3,201001001,month2
4,201001001,month2
...,...,...
15317,203908001,before_study
15318,203908001,month2
15319,203908001,month2
15320,203908001,month2


In [71]:
respeval

,SUBJID,Study_month
0,201001001,before_study
1,201001001,before_study
2,201001001,before_study
3,201001001,month2
4,201001001,month2
...,...,...
15317,203908001,before_study
15318,203908001,month2
15319,203908001,month2
15320,203908001,month2


# Adverse Events

In [72]:
ae = pd.read_csv("ae.csv")

In [73]:
ae.shape

(14842, 129)

In [74]:
ae['SUBJID'].unique()		

array([201001001, 201001002, 201001004, 201001005, 201001006, 201001007,
       201001008, 201001010, 201001011, 201001012, 201002001, 201002002,
       201002003, 201002004, 201002005, 201003001, 201003002, 201003003,
       201003004, 201003005, 201003006, 201004001, 201004002, 201005001,
       201005003, 201005004, 201005006, 201005008, 201005009, 201005011,
       201005012, 201007001, 201007003, 201008001, 201008002, 201008003,
       201008004, 201009001, 201010001, 201010002, 201101001, 201101002,
       201101003, 201101004, 201101005, 201101006, 201101007, 201101008,
       201101009, 201101010, 201101011, 201101012, 201101013, 201101015,
       201101016, 201101017, 201101018, 201101019, 201101020, 201102001,
       201102002, 201102003, 201102004, 201102005, 201102006, 201102007,
       201102008, 201102009, 201102010, 201103001, 201103002, 201103003,
       201103004, 201103005, 201103006, 201103007, 201103008, 201103009,
       201103010, 201103012, 201103013, 201103014, 

In [75]:
ae['AELLT'].nunique()

1527

In [76]:
ae['AEHLT'].nunique()		

380

In [77]:
ae['AESOC'].nunique()		

24

In [78]:
ae['AEINTCAT'].unique()		

array(['SKIN', nan, 'HAIR', 'NAIL', 'EYE', 'CHEILITIS'], dtype=object)

In [79]:
ae['AESEV'].unique()	

array(['Mild', 'Moderate', 'Severe', 'Life threatening', 'Fatal', nan],
      dtype=object)

In [80]:
ae['Duration'] = ae['AEENDYI'] - ae['AESTDYI']

In [81]:
columns_to_drop = ['STUDYID','B_METACT','AEACTCDS','AEPRCNT','SAELIFE','SITEID','AGE','SEXCD','SEX','RACCATCD','RACCAT','TRTDOSE','TRTFRQCD','TRTFREQ','TRTCD','TRT','ATRTCD','ATRT','PRBEVCDR','PRBEVR',
                   'PRBEVCDM','PRBEVM','PRBEVCD','PRBEV','PROXACDR','PROXAR','PROXACDM','PROXAM','PROXALCD','PROXAL','B_METANM','LIVRONCD','LIVRONLY',
                   'PADJYNCD','PRADJYN','DPADJUV','B_LDHYN','B_LDH2YN','B_LDHNM','BECOGICD','B_ECOGI','BECGICT','DIAGTYCD','DIAGTYPE','TUMCATCD',
                   'TUMCAT','LIVERMET','KRASCD','KRAS','EVALITT','EVALPP','EVALQOL','EVALEGFR','PHASECD','PHASE','PHSENDCD','PHASEND','B_ECOGCD',
                   'B_ECOG','TRTDUR','MEDDRA_V','AELINENO','AESTDYI','AEENDYI','AEENDY','AESTDY','AEDURI','AETERM','AELLT','AEHLGT','AEPT','AEHLT','AELLTCD','AEINTCAT','AENASTCD','AENASTER','AEINTMT']
ae.drop(columns=columns_to_drop, inplace=True)

In [82]:
ae.columns

Index(['SUBJID', 'AEANY', 'AESOC', 'AEDUR', 'AECONT', 'AENUMINF', 'AENUMCTX', 'AEPRIOR', 'AERASH', 'AEINTEYE', 'AEINTINF', 'AESEVCD', 'AESEV', 'AEREL', 'AERELCM', 'AEINFCTC', 'AEINFREP', 'AEINFPI', 'AECRDTOX', 'AEPULTOX', 'AEVASTOX', 'AEDIARR', 'AESTOMA', 'AECALCEM', 'AEMAGNES', 'AENEUTOX', 'AEHEMTOX', 'AEACT01', 'AEACT02', 'AEACT03', 'AEACT04', 'AEACT05', 'AEACT06', 'AEACT07', 'AEACT10', 'AEACT81', 'AEACT82', 'AEACT83', 'AEACT88', 'AESER', 'AEWDOS', 'AECDOS', 'AEIRWDOS', 'AEIRCDOS', 'AEFIWDOS', 'AEFICDOS', 'AEFBWDOS', 'AEFBCDOS', 'AELLWDOS', 'AELLCDOS', 'AEDLWDOS', 'AEDLCDOS', 'Duration'], dtype='object')

In [83]:
ae['AEANY_Y'] = ae['AEANY'].apply(lambda x: 1 if x == 'Y' else 0)
ae['AEANY_N'] = ae['AEANY'].apply(lambda x: 1 if x == 'N' else 0)

# Drop the original AEANY column if not needed
ae.drop(columns=['AEANY'], inplace=True)

In [84]:
columns_to_transform = [
    'AECONT', 'AEPRIOR', 'AERASH', 'AEINTEYE', 'AEINTINF', 'AEREL', 'AERELCM',
    'AEINFCTC', 'AEINFREP', 'AEINFPI', 'AECRDTOX', 'AEPULTOX', 'AEVASTOX',
    'AEDIARR', 'AESTOMA', 'AECALCEM', 'AEMAGNES', 'AENEUTOX', 'AEHEMTOX',
    'AEACT01', 'AEACT02', 'AEACT03', 'AEACT04', 'AEACT05', 'AEACT06',
    'AEACT07', 'AEACT10', 'AEACT81', 'AEACT82', 'AEACT83', 'AEACT88', 'AESER'
]

for col in columns_to_transform:
    if col in ae.columns:
        ae[f'{col}_Y'] = ae[col].apply(lambda x: 1 if x == 'Y' else 0)
        ae[f'{col}_N'] = ae[col].apply(lambda x: 1 if x == 'N' else 0)
        ae.drop(columns=[col], inplace=True)

ae


,SUBJID,AESOC,AEDUR,AENUMINF,AENUMCTX,AESEVCD,AESEV,AEWDOS,AECDOS,AEIRWDOS,AEIRCDOS,AEFIWDOS,AEFICDOS,AEFBWDOS,AEFBCDOS,AELLWDOS,AELLCDOS,AEDLWDOS,AEDLCDOS,Duration,AEANY_Y,AEANY_N,AECONT_Y,AECONT_N,AEPRIOR_Y,AEPRIOR_N,AERASH_Y,AERASH_N,AEINTEYE_Y,AEINTEYE_N,AEINTINF_Y,AEINTINF_N,AEREL_Y,AEREL_N,AERELCM_Y,AERELCM_N,AEINFCTC_Y,AEINFCTC_N,AEINFREP_Y,AEINFREP_N,AEINFPI_Y,AEINFPI_N,AECRDTOX_Y,AECRDTOX_N,AEPULTOX_Y,AEPULTOX_N,AEVASTOX_Y,AEVASTOX_N,AEDIARR_Y,AEDIARR_N,AESTOMA_Y,AESTOMA_N,AECALCEM_Y,AECALCEM_N,AEMAGNES_Y,AEMAGNES_N,AENEUTOX_Y,AENEUTOX_N,AEHEMTOX_Y,AEHEMTOX_N,AEACT01_Y,AEACT01_N,AEACT02_Y,AEACT02_N,AEACT03_Y,AEACT03_N,AEACT04_Y,AEACT04_N,AEACT05_Y,AEACT05_N,AEACT06_Y,AEACT06_N,AEACT07_Y,AEACT07_N,AEACT10_Y,AEACT10_N,AEACT81_Y,AEACT81_N,AEACT82_Y,AEACT82_N,AEACT83_Y,AEACT83_N,AEACT88_Y,AEACT88_N,AESER_Y,AESER_N
0,201001001,Skin and subcutaneous tissue disorders,NaN,1.0,1.0,1.0,Mild,6.000000,6.000000,179.903298,179.903298,2398.710643,2398.710643,399.785107,399.785107,NaN,NaN,399.785107,399.785107,NaN,1,0,1,0,0,1,1,0,1,0,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
1,201001001,Gastrointestinal disorders,NaN,1.0,1.0,1.0,Mild,6.000000,6.000000,179.903298,179.903298,2398.710643,2398.710643,399.785107,399.785107,NaN,NaN,399.785107,399.785107,203.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
2,201001001,Skin and subcutaneous tissue disorders,49.0,1.0,1.0,1.0,Mild,6.000000,6.000000,179.903298,179.903298,2398.710643,2398.710643,399.785107,399.785107,NaN,NaN,399.785107,399.785107,48.0,1,0,0,1,0,1,1,0,1,0,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
3,201001001,Skin and subcutaneous tissue disorders,NaN,11.0,11.0,1.0,Mild,6.000000,65.777778,181.091714,1982.449216,2416.611468,26442.364034,402.430451,4405.475769,NaN,NaN,401.774088,4404.163042,NaN,1,0,1,0,0,1,0,1,1,0,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
4,201001001,Infections and infestations,NaN,16.0,16.0,2.0,Moderate,6.075949,96.083523,181.273526,2887.702109,2416.846908,38511.736255,402.832837,6417.162747,NaN,NaN,402.307436,6413.226244,NaN,1,0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14837,203908001,Gastrointestinal disorders,5.0,0.0,3.0,1.0,Mild,NaN,NaN,184.475608,542.832708,2461.910843,7247.607028,409.200440,1204.101644,NaN,NaN,409.200440,1204.101644,4.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
14838,203908001,Blood and lymphatic system disorders,7.0,0.0,4.0,4.0,Life threatening,NaN,NaN,176.184041,719.016749,2392.037173,9639.644201,399.802247,1603.903891,NaN,NaN,399.802247,1603.903891,6.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
14839,203908001,Blood and lymphatic system disorders,5.0,0.0,4.0,2.0,Moderate,NaN,NaN,176.184041,719.016749,2392.037173,9639.644201,399.802247,1603.903891,NaN,NaN,399.802247,1603.903891,4.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1
14840,203908001,Gastrointestinal disorders,5.0,0.0,4.0,1.0,Mild,NaN,NaN,176.184041,719.016749,2392.037173,9639.644201,399.802247,1603.903891,NaN,NaN,399.802247,1603.903891,4.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,

In [85]:
ae['SUBJID'].nunique

<bound method IndexOpsMixin.nunique of 0        201001001
1        201001001
2        201001001
3        201001001
4        201001001
           ...    
14837    203908001
14838    203908001
14839    203908001
14840    203908001
14841    203908001
Name: SUBJID, Length: 14842, dtype: int64>

In [86]:
study_month = pd.merge(respeval,ae , on='SUBJID', how='inner')

In [87]:
study_month

,SUBJID,Study_month,AESOC,AEDUR,AENUMINF,AENUMCTX,AESEVCD,AESEV,AEWDOS,AECDOS,AEIRWDOS,AEIRCDOS,AEFIWDOS,AEFICDOS,AEFBWDOS,AEFBCDOS,AELLWDOS,AELLCDOS,AEDLWDOS,AEDLCDOS,Duration,AEANY_Y,AEANY_N,AECONT_Y,AECONT_N,AEPRIOR_Y,AEPRIOR_N,AERASH_Y,AERASH_N,AEINTEYE_Y,AEINTEYE_N,AEINTINF_Y,AEINTINF_N,AEREL_Y,AEREL_N,AERELCM_Y,AERELCM_N,AEINFCTC_Y,AEINFCTC_N,AEINFREP_Y,AEINFREP_N,AEINFPI_Y,AEINFPI_N,AECRDTOX_Y,AECRDTOX_N,AEPULTOX_Y,AEPULTOX_N,AEVASTOX_Y,AEVASTOX_N,AEDIARR_Y,AEDIARR_N,AESTOMA_Y,AESTOMA_N,AECALCEM_Y,AECALCEM_N,AEMAGNES_Y,AEMAGNES_N,AENEUTOX_Y,AENEUTOX_N,AEHEMTOX_Y,AEHEMTOX_N,AEACT01_Y,AEACT01_N,AEACT02_Y,AEACT02_N,AEACT03_Y,AEACT03_N,AEACT04_Y,AEACT04_N,AEACT05_Y,AEACT05_N,AEACT06_Y,AEACT06_N,AEACT07_Y,AEACT07_N,AEACT10_Y,AEACT10_N,AEACT81_Y,AEACT81_N,AEACT82_Y,AEACT82_N,AEACT83_Y,AEACT83_N,AEACT88_Y,AEACT88_N,AESER_Y,AESER_N
0,201001001,before_study,Skin and subcutaneous tissue disorders,NaN,1.0,1.0,1.0,Mild,6.000000,6.000000,179.903298,179.903298,2398.710643,2398.710643,399.785107,399.785107,NaN,NaN,399.785107,399.785107,NaN,1,0,1,0,0,1,1,0,1,0,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
1,201001001,before_study,Gastrointestinal disorders,NaN,1.0,1.0,1.0,Mild,6.000000,6.000000,179.903298,179.903298,2398.710643,2398.710643,399.785107,399.785107,NaN,NaN,399.785107,399.785107,203.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
2,201001001,before_study,Skin and subcutaneous tissue disorders,49.0,1.0,1.0,1.0,Mild,6.000000,6.000000,179.903298,179.903298,2398.710643,2398.710643,399.785107,399.785107,NaN,NaN,399.785107,399.785107,48.0,1,0,0,1,0,1,1,0,1,0,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
3,201001001,before_study,Skin and subcutaneous tissue disorders,NaN,11.0,11.0,1.0,Mild,6.000000,65.777778,181.091714,1982.449216,2416.611468,26442.364034,402.430451,4405.475769,NaN,NaN,401.774088,4404.163042,NaN,1,0,1,0,0,1,0,1,1,0,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
4,201001001,before_study,Infections and infestations,NaN,16.0,16.0,2.0,Moderate,6.075949,96.083523,181.273526,2887.702109,2416.846908,38511.736255,402.832837,6417.162747,NaN,NaN,402.307436,6413.226244,NaN,1,0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316711,203908001,month3,Gastrointestinal disorders,5.0,0.0,3.0,1.0,Mild,NaN,NaN,184.475608,542.832708,2461.910843,7247.607028,409.200440,1204.101644,NaN,NaN,409.200440,1204.101644,4.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
316712,203908001,month3,Blood and lymphatic system disorders,7.0,0.0,4.0,4.0,Life threatening,NaN,NaN,176.184041,719.016749,2392.037173,9639.644201,399.802247,1603.903891,NaN,NaN,399.802247,1603.903891,6.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1
316713,203908001,month3,Blood and lymphatic system disorders,5.0,0.0,4.0,2.0,Moderate,NaN,NaN,176.184041,719.016749,2392.037173,9639.644201,399.802247,1603.903891,NaN,NaN,399.802247,1603.903891,4.0,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0,1
316714,203908001,month3,Gastrointestinal disorders,5.0,0.0,4.0,1.0,Mild,NaN,NaN,176.184041,719.016749,2392.037173,9639.644201,399.802247,

In [88]:
df = study_month

# Month 1

In [89]:
df = df[df['Study_month'] == 'month1']
df.drop(columns=['Study_month'], inplace=True)




C:\Users\Dell\AppData\Local\Temp\ipykernel_11332\453308909.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['Study_month'], inplace=True)


In [90]:
df['SUBJID'].nunique()

39

In [91]:
unique_subject_ids = df['SUBJID'].unique()
unique_subject_ids

array([201005001, 201101002, 201101003, 201101006, 201101016, 201101018,
       201101020, 201102009, 201103004, 201103019, 201115003, 201222005,
       201223002, 201228002, 201228003, 201228006, 201304006, 201322011,
       201404010, 201419006, 201419008, 201501002, 201508001, 202002004,
       202002005, 203402005, 203405003, 203506008, 203514002, 203604008,
       203604009, 203605006, 203610009, 203611003, 203611004, 203611005,
       203705005, 203801002, 203903023], dtype=int64)

In [92]:
mean_df = df.groupby('SUBJID')[['AEWDOS','AEDUR','AENUMINF','AENUMCTX','AEWDOS','AECDOS','AEDLWDOS','AEDLCDOS','AEFIWDOS','AEIRWDOS','AEIRCDOS','AEFICDOS','AEFBWDOS','AEFBCDOS','AELLWDOS','AELLCDOS','AEDLWDOS','AEDLCDOS']].mean().reset_index()

In [93]:
mean_df.head()

,SUBJID,AEWDOS,AEDUR,AENUMINF,AENUMCTX,AEWDOS,AECDOS,AEDLWDOS,AEDLCDOS,AEFIWDOS,AEIRWDOS,AEIRCDOS,AEFICDOS,AEFBWDOS,AEFBCDOS,AELLWDOS,AELLCDOS,AEDLWDOS,AEDLCDOS
0,201005001,NaN,14.250000,0.0,5.500000,NaN,NaN,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,NaN,NaN,403.931339,2221.622362
1,201101002,NaN,30.866667,0.0,8.600000,NaN,NaN,NaN,NaN,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,NaN,NaN
2,201101003,6.0,117.500000,1.0,1.000000,6.0,6.0,NaN,NaN,2288.743729,171.655780,171.655780,2288.743729,381.457288,381.457288,190.728644,190.728644,NaN,NaN
3,201101006,NaN,18.000000,0.0,1.666667,NaN,NaN,NaN,NaN,2244.116537,168.308740,301.445738,4019.276513,369.054787,658.692570,200.031708,364.282265,NaN,NaN
4,201101016,NaN,31.100000,0.0,3.650000,NaN,NaN,NaN,NaN,2138.486013,160.386451,584.087786,7787.837153,356.414335,1297.972859,178.207168,648.986429,NaN,NaN


In [94]:
mean_df.fillna(0, inplace=True)

In [95]:
df_one_hot = pd.get_dummies(ae, columns=['AESOC', 'AESEV'], prefix=['AESOC', 'AESEV'])

# Group by SUBJID and AEPT, summing the one-hot encoded columns
ae = df_one_hot.groupby(['SUBJID','AEDUR'], as_index=False).sum()

In [96]:
ae.isnull().sum()

SUBJID                                                                       0
AEDUR                                                                        0
AENUMINF                                                                     0
AENUMCTX                                                                     0
AESEVCD                                                                      0
AEWDOS                                                                       0
AECDOS                                                                       0
AEIRWDOS                                                                     0
AEIRCDOS                                                                     0
AEFIWDOS                                                                     0
AEFICDOS                                                                     0
AEFBWDOS                                                                     0
AEFBCDOS                                            

In [97]:
subset_df = ae[['SUBJID','AESOC_Investigations', 'AESOC_Metabolism and nutrition disorders','AESOC_Blood and lymphatic system disorders','AESOC_Cardiac disorders','AESOC_Congenital, familial and genetic disorders','AESOC_Ear and labyrinth disorders','AESOC_Endocrine disorders','AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 'AESOC_Injury, poisoning and procedural complications', 'AESOC_Investigations', 'AESOC_Metabolism and nutrition disorders', 'AESOC_Musculoskeletal and connective tissue disorders', 'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe']]
                        
                                     
                


In [98]:
subset_df.drop_duplicates(keep='first')

,SUBJID,AESOC_Investigations,AESOC_Metabolism and nutrition disorders,AESOC_Blood and lymphatic system disorders,AESOC_Cardiac disorders,"AESOC_Congenital, familial and genetic disorders",AESOC_Ear and labyrinth disorders,AESOC_Endocrine disorders,AESOC_Eye disorders,AESOC_Gastrointestinal disorders,AESOC_General disorders and administration site conditions,AESOC_Hepatobiliary disorders,AESOC_Immune system disorders,AESOC_Infections and infestations,"AESOC_Injury, poisoning and procedural complications",AESOC_Investigations,AESOC_Metabolism and nutrition disorders,AESOC_Musculoskeletal and connective tissue disorders,"AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)",AESOC_Nervous system disorders,AESOC_Psychiatric disorders,AESOC_Renal and urinary disorders,AESOC_Reproductive system and breast disorders,"AESOC_Respiratory, thoracic and mediastinal disorders",AESOC_Skin and subcutaneous tissue disorders,AESOC_Surgical and medical procedures,AESOC_Vascular disorders,AESEV_Fatal,AESEV_Life threatening,AESEV_Mild,AESEV_Moderate,AESEV_Severe
0,201001001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0
1,201001002,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0
2,201001002,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
3,201001004,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,201001004,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7233,203906006,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0
7234,203908001,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
7235,203908001,0,0,1,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,1,0
7236,203908001,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


In [99]:
subset_df.shape

(7238, 32)

In [100]:
merged_df = pd.merge(mean_df,subset_df, on='SUBJID', how='inner')

In [101]:
merged_df

,SUBJID,AEWDOS,AEDUR,AENUMINF,AENUMCTX,AEWDOS,AECDOS,AEDLWDOS,AEDLCDOS,AEFIWDOS,AEIRWDOS,AEIRCDOS,AEFICDOS,AEFBWDOS,AEFBCDOS,AELLWDOS,AELLCDOS,AEDLWDOS,AEDLCDOS,AESOC_Investigations,AESOC_Metabolism and nutrition disorders,AESOC_Blood and lymphatic system disorders,AESOC_Cardiac disorders,"AESOC_Congenital, familial and genetic disorders",AESOC_Ear and labyrinth disorders,AESOC_Endocrine disorders,AESOC_Eye disorders,AESOC_Gastrointestinal disorders,AESOC_General disorders and administration site conditions,AESOC_Hepatobiliary disorders,AESOC_Immune system disorders,AESOC_Infections and infestations,"AESOC_Injury, poisoning and procedural complications",AESOC_Investigations,AESOC_Metabolism and nutrition disorders,AESOC_Musculoskeletal and connective tissue disorders,"AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)",AESOC_Nervous system disorders,AESOC_Psychiatric disorders,AESOC_Renal and urinary disorders,AESOC_Reproductive system and breast disorders,"AESOC_Respiratory, thoracic and mediastinal disorders",AESOC_Skin and subcutaneous tissue disorders,AESOC_Surgical and medical procedures,AESOC_Vascular disorders,AESEV_Fatal,AESEV_Life threatening,AESEV_Mild,AESEV_Moderate,AESEV_Severe
0,201005001,0.000000,14.250000,0.000000,5.500000,0.000000,0.000000,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,403.931339,2221.622362,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,201005001,0.000000,14.250000,0.000000,5.500000,0.000000,0.000000,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,403.931339,2221.622362,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,201005001,0.000000,14.250000,0.000000,5.500000,0.000000,0.000000,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,403.931339,2221.622362,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,201005001,0.000000,14.250000,0.000000,5.500000,0.000000,0.000000,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,403.931339,2221.622362,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,201101002,0.000000,30.866667,0.000000,8.600000,0.000000,0.000000,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0.000000,0.000000,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,4,0,0
5,201101002,0.000000,30.866667,0.000000,8.600000,0.000000,0.000000,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0.000000,0.000000,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
6,201101002,0.000000,30.866667,0.000000,8.600000,0.000000,0.000000,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0
7,201101002,0.000000,30.866667,0.000000,8.600000,0.000000,0.000000,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0.000000,0.000000,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,2,0
8,201101002,0.000000,30.866667,0.000000,8.600000,0.000000,0.000000,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0
9,201101002,0.000000,30.866667,0.000000,8.600000,0.000000,0.000000,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [102]:
merged_df.shape

(330, 50)

In [103]:
merged_df['AEWDOS'].nunique()

AEWDOS    12
AEWDOS    12
dtype: int64

In [104]:
ae['AEANY_N'].unique()

array([0], dtype=int64)

In [105]:
df.to_csv('month1.csv', index=False)

In [106]:
mean_df.head()


,SUBJID,AEWDOS,AEDUR,AENUMINF,AENUMCTX,AEWDOS,AECDOS,AEDLWDOS,AEDLCDOS,AEFIWDOS,AEIRWDOS,AEIRCDOS,AEFICDOS,AEFBWDOS,AEFBCDOS,AELLWDOS,AELLCDOS,AEDLWDOS,AEDLCDOS
0,201005001,0.0,14.250000,0.0,5.500000,0.0,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,403.931339,2221.622362
1,201101002,0.0,30.866667,0.0,8.600000,0.0,0.0,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0.000000,0.000000
2,201101003,6.0,117.500000,1.0,1.000000,6.0,6.0,0.000000,0.000000,2288.743729,171.655780,171.655780,2288.743729,381.457288,381.457288,190.728644,190.728644,0.000000,0.000000
3,201101006,0.0,18.000000,0.0,1.666667,0.0,0.0,0.000000,0.000000,2244.116537,168.308740,301.445738,4019.276513,369.054787,658.692570,200.031708,364.282265,0.000000,0.000000
4,201101016,0.0,31.100000,0.0,3.650000,0.0,0.0,0.000000,0.000000,2138.486013,160.386451,584.087786,7787.837153,356.414335,1297.972859,178.207168,648.986429,0.000000,0.000000


In [107]:
ae.columns

Index(['SUBJID', 'AEDUR', 'AENUMINF', 'AENUMCTX', 'AESEVCD', 'AEWDOS', 'AECDOS', 'AEIRWDOS', 'AEIRCDOS', 'AEFIWDOS',
       ...
       'AESOC_Reproductive system and breast disorders', 'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'], dtype='object', length=113)

# XGBOOST

## AESOC_1

In [108]:
import xgboost 

In [109]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [110]:
SEED = 2222
EARLY_STOP_NUM = 8

In [111]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [112]:
numeric_columns = merged_df[numeric_columns]

In [113]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [114]:
numeric_columns.head()

,SUBJID,AEWDOS,AEDUR,AENUMINF,AENUMCTX,AECDOS,AEDLWDOS,AEDLCDOS,AEFIWDOS,AEIRWDOS,AEIRCDOS,AEFICDOS,AEFBWDOS,AEFBCDOS,AELLWDOS,AELLCDOS,AESOC_Investigations,AESOC_Metabolism and nutrition disorders,AESOC_Blood and lymphatic system disorders,AESOC_Cardiac disorders,"AESOC_Congenital, familial and genetic disorders",AESOC_Ear and labyrinth disorders,AESOC_Endocrine disorders,AESOC_Eye disorders,AESOC_Gastrointestinal disorders,AESOC_General disorders and administration site conditions,AESOC_Hepatobiliary disorders,AESOC_Immune system disorders,AESOC_Infections and infestations,"AESOC_Injury, poisoning and procedural complications",AESOC_Musculoskeletal and connective tissue disorders,"AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)",AESOC_Nervous system disorders,AESOC_Psychiatric disorders,AESOC_Renal and urinary disorders,AESOC_Reproductive system and breast disorders,"AESOC_Respiratory, thoracic and mediastinal disorders",AESOC_Skin and subcutaneous tissue disorders,AESOC_Surgical and medical procedures,AESOC_Vascular disorders,AESEV_Fatal,AESEV_Life threatening,AESEV_Mild,AESEV_Moderate,AESEV_Severe
0,201005001,0.0,14.250000,0.0,5.5,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,201005001,0.0,14.250000,0.0,5.5,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,201005001,0.0,14.250000,0.0,5.5,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,201005001,0.0,14.250000,0.0,5.5,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,201101002,0.0,30.866667,0.0,8.6,0.0,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,4,0,0


In [115]:
drop_columns = [
    'AESOC_Metabolism and nutrition disorders', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [116]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Investigations']


In [117]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


Missing columns: []


In [118]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

[0]	validation_0-rmse:0.11336
[1]	validation_0-rmse:0.07993
[2]	validation_0-rmse:0.05099
[3]	validation_0-rmse:0.03256
[4]	validation_0-rmse:0.01999
[5]	validation_0-rmse:0.01172
[6]	validation_0-rmse:0.00647
[7]	validation_0-rmse:0.00499
[8]	validation_0-rmse:0.00251
[9]	validation_0-rmse:0.00076
[10]	validation_0-rmse:0.00035
[11]	validation_0-rmse:0.00021
[12]	validation_0-rmse:0.00034
[13]	validation_0-rmse:0.00033
[14]	validation_0-rmse:0.00035
[15]	validation_0-rmse:0.00034
[16]	validation_0-rmse:0.00038
[17]	validation_0-rmse:0.00040
[18]	validation_0-rmse:0.00040
Validation RMSE: 0.0002083428042636292
Validation MSE: 4.340672408843291e-08


C:\Users\Dell\anaconda3\Lib\site-packages\xgboost\sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [119]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

array([0.00021067, 0.00021067, 0.00021067, 0.00021067, 0.00021067],
      dtype=float32)

# Feature Selection by Random Forest

In [120]:
from sklearn.ensemble import RandomForestClassifier

In [121]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

RandomForestClassifier(random_state=42)

In [122]:
importances = rf.feature_importances_

In [123]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [124]:
feature_importances_df.head(20)

,Feature,Importance
16,AESOC_Investigations,0.745950
3,AENUMINF,0.033531
10,AEIRCDOS,0.033514
5,AECDOS,0.032730
8,AEFIWDOS,0.029243
12,AEFBWDOS,0.023210
7,AEDLCDOS,0.015479
9,AEIRWDOS,0.014527
6,AEDLWDOS,0.014304
0,SUBJID,0.011507


## AESOC_2

In [125]:
import xgboost 

In [126]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [127]:
SEED = 2222
EARLY_STOP_NUM = 8

In [128]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [129]:
numeric_columns = merged_df[numeric_columns]

In [130]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [131]:
numeric_columns.head()

,SUBJID,AEWDOS,AEDUR,AENUMINF,AENUMCTX,AECDOS,AEDLWDOS,AEDLCDOS,AEFIWDOS,AEIRWDOS,AEIRCDOS,AEFICDOS,AEFBWDOS,AEFBCDOS,AELLWDOS,AELLCDOS,AESOC_Investigations,AESOC_Metabolism and nutrition disorders,AESOC_Blood and lymphatic system disorders,AESOC_Cardiac disorders,"AESOC_Congenital, familial and genetic disorders",AESOC_Ear and labyrinth disorders,AESOC_Endocrine disorders,AESOC_Eye disorders,AESOC_Gastrointestinal disorders,AESOC_General disorders and administration site conditions,AESOC_Hepatobiliary disorders,AESOC_Immune system disorders,AESOC_Infections and infestations,"AESOC_Injury, poisoning and procedural complications",AESOC_Musculoskeletal and connective tissue disorders,"AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)",AESOC_Nervous system disorders,AESOC_Psychiatric disorders,AESOC_Renal and urinary disorders,AESOC_Reproductive system and breast disorders,"AESOC_Respiratory, thoracic and mediastinal disorders",AESOC_Skin and subcutaneous tissue disorders,AESOC_Surgical and medical procedures,AESOC_Vascular disorders,AESEV_Fatal,AESEV_Life threatening,AESEV_Mild,AESEV_Moderate,AESEV_Severe
0,201005001,0.0,14.250000,0.0,5.5,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,201005001,0.0,14.250000,0.0,5.5,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,201005001,0.0,14.250000,0.0,5.5,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,201005001,0.0,14.250000,0.0,5.5,0.0,403.931339,2221.622362,2189.989185,180.065777,990.361776,13139.935112,362.564876,2180.255900,0.000000,0.000000,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,201101002,0.0,30.866667,0.0,8.6,0.0,0.000000,0.000000,2377.582923,178.152783,1540.070072,20555.313788,396.612410,3432.314209,198.029343,1712.557901,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,4,0,0


In [132]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [133]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Metabolism and nutrition disorders']


In [134]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


Missing columns: []


In [135]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

[0]	validation_0-rmse:0.16814
[1]	validation_0-rmse:0.10285
[2]	validation_0-rmse:0.06293
[3]	validation_0-rmse:0.03852
[4]	validation_0-rmse:0.02270
[5]	validation_0-rmse:0.01391
[6]	validation_0-rmse:0.00854
[7]	validation_0-rmse:0.00523
[8]	validation_0-rmse:0.00321
[9]	validation_0-rmse:0.00197
[10]	validation_0-rmse:0.00121
[11]	validation_0-rmse:0.00074
[12]	validation_0-rmse:0.00046
[13]	validation_0-rmse:0.00028
[14]	validation_0-rmse:0.00017
[15]	validation_0-rmse:0.00011
[16]	validation_0-rmse:0.00007
[17]	validation_0-rmse:0.00004
[18]	validation_0-rmse:0.00004
[19]	validation_0-rmse:0.00004
[20]	validation_0-rmse:0.00004
[21]	validation_0-rmse:0.00004
[22]	validation_0-rmse:0.00004
[23]	validation_0-rmse:0.00004
[24]	validation_0-rmse:0.00004
[25]	validation_0-rmse:0.00004
[26]	validation_0-rmse:0.00004
[27]	validation_0-rmse:0.00004
[28]	validation_0-rmse:0.00004
[29]	validation_0-rmse:0.00004
[30]	validation_0-rmse:0.00004
Validation RMSE: 3.86377671068072e-05
Validation 

C:\Users\Dell\anaconda3\Lib\site-packages\xgboost\sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [136]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

array([1.736592e-05, 1.736592e-05, 1.736592e-05, 1.736592e-05,
       1.736592e-05], dtype=float32)

# Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

# AESOC - 3

In [ ]:
import xgboost

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations','AESOC_Metabolism and nutrition disorders' ,
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Blood and lymphatic system disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 4


In [ ]:
import xgboost 

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders','AESOC_Metabolism and nutrition disorders'
    , 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Cardiac disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 5

In [ ]:
import xgboost 

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders','AESOC_Metabolism and nutrition disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Congenital, familial and genetic disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 6

In [ ]:
import xgboost 

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders','AESOC_Metabolism and nutrition disorders' ,
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Ear and labyrinth disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 7

In [ ]:
import xgboost 

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders','AESOC_Metabolism and nutrition disorders'
    , 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Endocrine disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 8

In [ ]:
import xgboost

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders','AESOC_Metabolism and nutrition disorders' , 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Eye disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 9

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Metabolism and nutrition disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Gastrointestinal disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 10

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 'AESOC_Metabolism and nutrition disorders'
    , 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_General disorders and administration site conditions']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 11

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Metabolism and nutrition disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Hepatobiliary disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 12

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
  'AESOC_Metabolism and nutrition disorders' , 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns[ 'AESOC_Immune system disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 13

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Metabolism and nutrition disorders', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Infections and infestations']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 14

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 'AESOC_Metabolism and nutrition disorders'
    , 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Injury, poisoning and procedural complications']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 15

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications','AESOC_Metabolism and nutrition disorders' , 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Musculoskeletal and connective tissue disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 16

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Metabolism and nutrition disorders', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 17

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)','AESOC_Metabolism and nutrition disorders' , 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Nervous system disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 18

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    , 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 'AESOC_Metabolism and nutrition disorders'
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Psychiatric disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 19

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
   'AESOC_Metabolism and nutrition disorders' , 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Respiratory, thoracic and mediastinal disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 20

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders','AESOC_Metabolism and nutrition disorders' , 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Skin and subcutaneous tissue disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 21

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Metabolism and nutrition disorders', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Surgical and medical procedures']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 22

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures','AESOC_Metabolism and nutrition disorders' , 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Vascular disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 23

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Metabolism and nutrition disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 24

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Metabolism and nutrition disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 2

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Metabolism and nutrition disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 2

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Metabolism and nutrition disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    

# AESOC 2

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
SEED = 2222
EARLY_STOP_NUM = 8

In [ ]:
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
numeric_columns = merged_df[numeric_columns]

In [ ]:
numeric_columns = numeric_columns.loc[:, ~numeric_columns.columns.duplicated()]

In [ ]:
numeric_columns.head()

In [ ]:
drop_columns = [
    'AESOC_Investigations', 'AESOC_Blood and lymphatic system disorders',
    'AESOC_Cardiac disorders', 'AESOC_Congenital, familial and genetic disorders', 'AESOC_Ear and labyrinth disorders',
    'AESOC_Endocrine disorders', 'AESOC_Eye disorders', 'AESOC_Gastrointestinal disorders', 
    'AESOC_General disorders and administration site conditions', 'AESOC_Hepatobiliary disorders', 
    'AESOC_Immune system disorders', 'AESOC_Infections and infestations', 
    'AESOC_Injury, poisoning and procedural complications', 'AESOC_Musculoskeletal and connective tissue disorders', 
    'AESOC_Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'AESOC_Nervous system disorders', 
    'AESOC_Psychiatric disorders', 'AESOC_Renal and urinary disorders', 'AESOC_Reproductive system and breast disorders', 
    'AESOC_Respiratory, thoracic and mediastinal disorders', 'AESOC_Skin and subcutaneous tissue disorders', 
    'AESOC_Surgical and medical procedures', 'AESOC_Vascular disorders', 'AESEV_Fatal', 
    'AESEV_Life threatening', 'AESEV_Mild', 'AESEV_Moderate', 'AESEV_Severe'
]


In [ ]:
X = numeric_columns.drop(columns=drop_columns, errors='ignore')
y = numeric_columns['AESOC_Metabolism and nutrition disorders']


In [ ]:
missing_columns = [col for col in drop_columns if col not in numeric_columns.columns]
print("Missing columns:", missing_columns)


In [ ]:
X_train_val1, X_test1, y_train_val1, y_test1 = train_test_split(X, y,test_size = 0.10, random_state=SEED)
X_train1, X_val1, y_train1, y_val1 = train_test_split(X_train_val1, y_train_val1,test_size = 0.30, random_state=SEED)

xgb_reg = xgboost.XGBRegressor(n_estimators=150, learning_rate=0.4, gamma=0, subsample=0.75,
                           colsample_bytree=1, max_depth=7,random_state=SEED)
xgb_reg.fit(X_train1, y_train1,eval_set=[(X_val1, y_val1)], early_stopping_rounds=EARLY_STOP_NUM)
y_pred = xgb_reg.predict(X_val1)
val_rmse = mean_squared_error(y_val1, y_pred,squared=False)  
print("Validation RMSE:", val_rmse) 
val_mse = mean_squared_error(y_val1, y_pred,squared=True)  
print("Validation MSE:", val_mse)

In [ ]:
pred_test1 = xgb_reg.predict(X_test1)
pred_test1[:5]

## Feature Selection by Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

In [ ]:
importances = rf.feature_importances_

In [ ]:
# Create a dataframe for feature importances
feature_importances_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importances_df = feature_importances_df.sort_values(by='Importance', ascending=False)

In [ ]:
feature_importances_df.head(20)

## Confusion_matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

report = classification_report(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
    